# 06 Baseline Stress Test (Dense Discrete-n Main Path)

This is a pre-neural-network stress test using lightweight tabular baselines.

Main protocols (headline):
- LOAO: Leave-One-a-Out
- LOARO: Leave-One-Aspect-Ratio-Out

Targets (in required order):
1. `E0`
2. `dE1`
3. `dE3`
4. `dE2` (absolute metrics are primary due to near-degeneracy sensitivity)

No neural networks, no production claims.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.model import BASELINE_MODEL_NAMES, build_baseline_model, run_baseline_stress_test, regression_metrics

In [ ]:
data_path = PROJECT_ROOT / "data" / "superellipse_discrete_n_dense_dataset.npz"
data = np.load(data_path)

X_all = np.column_stack([data["a"], data["aspect_ratio"]]).astype(float)
n_all = data["n"].astype(float)
a_all = data["a"].astype(float)
r_all = data["aspect_ratio"].astype(float)

targets = {
    "E0": data["E0"].astype(float),
    "dE1": data["dE1"].astype(float),
    "dE3": data["dE3"].astype(float),
    "dE2": data["dE2"].astype(float),
}

print(f"Loaded: {data_path}")
print(f"Samples: {X_all.shape[0]}")
print(f"n classes: {np.unique(n_all)}")

In [ ]:
def format_table(rows: list[dict[str, object]], cols: list[str]) -> str:
    """Simple monospace table formatter for compact notebook output."""
    col_widths = {c: max(len(c), max(len(str(r[c])) for r in rows)) for c in cols}
    header = " | ".join(c.ljust(col_widths[c]) for c in cols)
    sep = "-+-".join("-" * col_widths[c] for c in cols)
    lines = [header, sep]
    for r in rows:
        lines.append(" | ".join(str(r[c]).ljust(col_widths[c]) for c in cols))
    return "\n".join(lines)


def compact_metric(v: float) -> str:
    return f"{v:.4e}"

In [ ]:
# Stress-test per fixed n class.
results_by_class: dict[float, dict[str, dict[str, dict[str, object]]]] = {}

for n_val in np.unique(n_all):
    mask = np.isclose(n_all, n_val)
    X = X_all[mask]
    a_vals = a_all[mask]
    r_vals = r_all[mask]

    results_by_class[float(n_val)] = {}
    for target_name, y_full in targets.items():
        y = y_full[mask]
        results_by_class[float(n_val)][target_name] = run_baseline_stress_test(
            X=X,
            y=y,
            a_values=a_vals,
            aspect_ratio_values=r_vals,
            model_names=BASELINE_MODEL_NAMES,
        )

print("Stress-test evaluation complete.")

In [ ]:
# Verbose per-model printout removed to keep notebook output concise.
print("Detailed per-model raw blocks are intentionally omitted.")
print("Use the compact audit summary tables below for decisions.")

In [ ]:
# Visual held-out slice summary (example): LOAO MAE by held-out a for E0, poly2_ridge.
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)
axes = axes.ravel()

for ax, n_val in zip(axes, sorted(results_by_class)):
    per_group = results_by_class[n_val]["E0"]["poly2_ridge"]["LOAO"]["per_group"]
    groups = [d["group"] for d in per_group]
    maes = [d["mae"] for d in per_group]
    ax.bar(groups, maes)
    ax.set_title(f"n={n_val}")
    ax.set_xlabel("held-out a")
    ax.set_ylabel("LOAO MAE (E0)")
    ax.grid(True, alpha=0.25)

plt.suptitle("Held-out slice error summary: poly2_ridge, target E0, protocol LOAO")
plt.tight_layout()
plt.show()

In [ ]:
# Diagnostic-only full-fit residual summary (NOT a claim metric).
print("Diagnostic-only full-fit residuals (same data used for fit/eval):")
for n_val in sorted(np.unique(n_all)):
    mask = np.isclose(n_all, n_val)
    X = X_all[mask]
    y = targets["E0"][mask]
    model = build_baseline_model("ridge")
    model.fit(X, y)
    pred = model.predict(X)
    m = regression_metrics(y, pred)
    print(f"- n={n_val}: MAE={m.mae:.4e}, RMSE={m.rmse:.4e}, MaxAE={m.max_abs_error:.4e}")

In [ ]:
# Compact scientific audit summary tables (headline outputs).
eps_dE2 = 5e-3
print(f"Using eps_dE2 = {eps_dE2:.1e} for guarded near-zero dE2 diagnostic.")


def build_protocol_summary_rows(protocol: str) -> list[dict[str, object]]:
    rows: list[dict[str, object]] = []
    for n_val in sorted(results_by_class):
        class_mask = np.isclose(n_all, n_val)
        for target_name in ["E0", "dE1", "dE3", "dE2"]:
            y_true_class = targets[target_name][class_mask]
            median_abs_scale = float(np.median(np.abs(y_true_class)))

            best_model = None
            best_metrics = None
            best_mae = np.inf
            for model_name in BASELINE_MODEL_NAMES:
                overall = results_by_class[n_val][target_name][model_name][protocol]["overall"]
                if overall["mae"] < best_mae:
                    best_mae = overall["mae"]
                    best_model = model_name
                    best_metrics = overall

            assert best_model is not None and best_metrics is not None
            ratio = (
                float(best_metrics["max_abs_error"] / median_abs_scale)
                if median_abs_scale > 0
                else np.nan
            )

            near_zero_frac = "-"
            if target_name == "dE2":
                near_zero_frac = f"{float(np.mean(np.abs(y_true_class) < eps_dE2)):.3f}"

            rows.append(
                {
                    "n": f"{n_val:.1f}",
                    "target": target_name,
                    "protocol": protocol,
                    "best_model": best_model,
                    "MAE": compact_metric(float(best_metrics["mae"])),
                    "RMSE": compact_metric(float(best_metrics["rmse"])),
                    "MaxAE": compact_metric(float(best_metrics["max_abs_error"])),
                    "median|y|": compact_metric(median_abs_scale),
                    "MaxAE/median|y|": compact_metric(ratio),
                    "frac(|dE2|<eps)": near_zero_frac,
                }
            )
    return rows


rows_loao = build_protocol_summary_rows("LOAO")
rows_loaro = build_protocol_summary_rows("LOARO")

cols = [
    "n",
    "target",
    "protocol",
    "best_model",
    "MAE",
    "RMSE",
    "MaxAE",
    "median|y|",
    "MaxAE/median|y|",
    "frac(|dE2|<eps)",
]

print("\n" + "=" * 120)
print("Compact summary table: LOAO")
print(format_table([r for r in rows_loao], cols))

print("\n" + "=" * 120)
print("Compact summary table: LOARO")
print(format_table([r for r in rows_loaro], cols))

In [ ]:
# Optional CSV export for audit traceability.
import csv

reports_dir = PROJECT_ROOT / "data"
reports_dir.mkdir(parents=True, exist_ok=True)

loao_csv = reports_dir / "baseline_stress_summary_loao.csv"
loaro_csv = reports_dir / "baseline_stress_summary_loaro.csv"

for path, rows in [(loao_csv, rows_loao), (loaro_csv, rows_loaro)]:
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=cols)
        writer.writeheader()
        writer.writerows(rows)

print(f"Saved: {loao_csv}")
print(f"Saved: {loaro_csv}")